In [1]:
import os
import numpy as np
import pandas as pd
import pydicom
import matplotlib.pyplot as plt
from skimage.feature import local_binary_pattern, hog
from skimage.transform import resize
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, f1_score, balanced_accuracy_score,
    confusion_matrix, roc_curve
)
import warnings
warnings.filterwarnings("ignore")

SEED = 42
N_PACIENTES = 150
N_SLICES_POR_PACIENTE = 15
TAMANHO_IMAGEM = 128
WINDOW_CENTER = 400
WINDOW_WIDTH = 1800

base_competicao = "/kaggle/input/competitions/rsna-2022-cervical-spine-fracture-detection"
base_imagens = os.path.join(base_competicao, "train_images")
print("Configuração carregada.")

Configuração carregada.


In [2]:
train = pd.read_csv(os.path.join(base_competicao, "train.csv"))
print(f"Total de pacientes: {len(train)}")
print(train["patient_overall"].value_counts())
print(train["patient_overall"].value_counts(normalize=True))

Total de pacientes: 2019
patient_overall
0    1058
1     961
Name: count, dtype: int64
patient_overall
0    0.524022
1    0.475978
Name: proportion, dtype: float64


In [3]:
amostra, _ = train_test_split(
    train, train_size=N_PACIENTES, stratify=train["patient_overall"], random_state=SEED
)
treino_ids, val_ids = train_test_split(
    amostra, test_size=0.2, stratify=amostra["patient_overall"], random_state=SEED
)
treino_ids = treino_ids.copy()
val_ids = val_ids.copy()
treino_ids["split"] = "train"
val_ids["split"] = "val"
particao = pd.concat([treino_ids, val_ids]).reset_index(drop=True)
particao.to_csv("/kaggle/working/particao_congelada.csv", index=False)
print(f"Amostra: {len(amostra)} | Treino: {len(treino_ids)} | Validação: {len(val_ids)}")

Amostra: 150 | Treino: 120 | Validação: 30


In [4]:
def aplicar_janela(img_hu, center, width):
    minimo = center - width / 2
    maximo = center + width / 2
    img = np.clip(img_hu, minimo, maximo)
    return (img - minimo) / (maximo - minimo)

def extrair_histograma_slice(img):
    feats = {}
    hist, _ = np.histogram(img, bins=10, range=(0, 1))
    hist = hist / (hist.sum() + 1e-8)
    for i, v in enumerate(hist):
        feats[f"hist_{i}"] = v
    return feats

def processar_paciente_hist(study_uid):
    pasta = os.path.join(base_imagens, study_uid)
    if not os.path.exists(pasta):
        return None
    arquivos = sorted(os.listdir(pasta))
    if len(arquivos) <= N_SLICES_POR_PACIENTE:
        selecionados = arquivos
    else:
        indices = np.linspace(0, len(arquivos) - 1, N_SLICES_POR_PACIENTE).astype(int)
        selecionados = [arquivos[i] for i in indices]

    features_slices = []
    for arq in selecionados:
        try:
            ds = pydicom.dcmread(os.path.join(pasta, arq))
            slope = float(getattr(ds, "RescaleSlope", 1))
            intercept = float(getattr(ds, "RescaleIntercept", 0))
            img_hu = ds.pixel_array.astype(float) * slope + intercept
            img_janela = aplicar_janela(img_hu, WINDOW_CENTER, WINDOW_WIDTH)
            img_resized = resize(img_janela, (TAMANHO_IMAGEM, TAMANHO_IMAGEM), anti_aliasing=True)
            features_slices.append(extrair_histograma_slice(img_resized))
        except Exception:
            continue

    if not features_slices:
        return None
    df_slices = pd.DataFrame(features_slices)
    agregado = {}
    for col in df_slices.columns:
        agregado[f"{col}_mean"] = df_slices[col].mean()
        agregado[f"{col}_std"] = df_slices[col].std()
    return agregado

print("Funções de pré-processamento (Semana 2: apenas histograma) definidas.")

Funções de pré-processamento (Semana 2: apenas histograma) definidas.


In [5]:
linhas = []
for idx, row in particao.iterrows():
    feats = processar_paciente_hist(row["StudyInstanceUID"])
    if feats is not None:
        feats["StudyInstanceUID"] = row["StudyInstanceUID"]
        feats["patient_overall"] = row["patient_overall"]
        feats["split"] = row["split"]
        linhas.append(feats)
    if (idx + 1) % 20 == 0:
        print(f"{idx + 1}/{len(particao)} pacientes processados")

df_features_semana2 = pd.DataFrame(linhas)
df_features_semana2.to_csv("/kaggle/working/features_semana2_histograma.csv", index=False)
print(f"Concluído! Shape: {df_features_semana2.shape}")

20/150 pacientes processados
40/150 pacientes processados
60/150 pacientes processados
80/150 pacientes processados
100/150 pacientes processados
120/150 pacientes processados
140/150 pacientes processados
Concluído! Shape: (113, 23)


In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, f1_score, balanced_accuracy_score

colunas_meta = ["StudyInstanceUID", "patient_overall", "split"]
colunas_features = [c for c in df_features_semana2.columns if c not in colunas_meta]

treino = df_features_semana2[df_features_semana2["split"] == "train"]
val = df_features_semana2[df_features_semana2["split"] == "val"]
X_treino = treino[colunas_features].fillna(0)
y_treino = treino["patient_overall"]
X_val = val[colunas_features].fillna(0)
y_val = val["patient_overall"]

print("=== BASELINE TRIVIAL ===")
baseline = DummyClassifier(strategy="most_frequent", random_state=SEED)
baseline.fit(X_treino, y_treino)
pred_baseline = baseline.predict(X_val)
print("Acurácia balanceada:", balanced_accuracy_score(y_val, pred_baseline))
print("F1:", f1_score(y_val, pred_baseline, zero_division=0))

print("\n=== PRIMEIRO CLASSIFICADOR (Regressão Logística, apenas histograma) ===")
modelo = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEED, class_weight="balanced"))
])
modelo.fit(X_treino, y_treino)
proba = modelo.predict_proba(X_val)[:, 1]
pred = modelo.predict(X_val)
print("AUC-ROC:", roc_auc_score(y_val, proba))
print("F1:", f1_score(y_val, pred, zero_division=0))
print("Acurácia balanceada:", balanced_accuracy_score(y_val, pred))

=== BASELINE TRIVIAL ===
Acurácia balanceada: 0.5
F1: 0.6470588235294118

=== PRIMEIRO CLASSIFICADOR (Regressão Logística, apenas histograma) ===
AUC-ROC: 0.5
F1: 0.4444444444444444
Acurácia balanceada: 0.5568181818181819
